# Module 7: Adversarial Attacks & the Secure AI/ML Lifecycle

---

## Learning Objectives

By the end of this module, you will be able to:

- Explain **why adversarial machine learning is different from ordinary software bugs** — there
  is an optimiser on the other side, choosing inputs specifically to defeat your model.
- Place any attack in the standard taxonomy — **evasion, poisoning, inference, extraction** —
  and map it with **MITRE ATLAS**, the ATT&CK-style knowledge base for AI systems.
- **Evade** a model at test time: craft a minimal change to an input that flips the
  prediction, and connect the ease of the attack to what the model actually learned.
- Distinguish **feature-space from problem-space** attacks, and state honestly what a
  feature-space result does and does not prove about a real adversary.
- **Poison** a training set: install a trigger backdoor with a fraction of a percent of the
  data, and show it is invisible to anyone reading the headline metric.
- Show that a trained model is both a **privacy surface** (it leaks its training data) and
  **stealable IP** (it can be cloned through its own API).
- **Defend**: apply adversarial training, measure the robustness it buys and the clean
  accuracy it costs, and state the limit — it defends the attack you trained against, not all
  attacks.
- Threat-model an ML pipeline across its **lifecycle** and argue for defence in depth.

---

## The model you were proud of breaks in ten lines

For six modules you built detectors and learned to distrust their numbers. Every surprise so
far came from the *data* — a leaky split, a benchmark with contradictory labels, a corpus that
measured the wrong thing. The models themselves were at worst honest and underfit. Nobody was
attacking them.

This module introduces the adversary. That single change breaks almost everything.

An ordinary bug is a mistake the software makes on its own. An **adversarial** failure is one
that someone *searches for* — an attacker who can query your model, or influence its training
data, and who is optimising, deliberately, for the input that makes it fail. Your 88% F1 was
measured against nature. Nature does not adapt. An attacker does, and the number that describes
your model against a passive test set says almost nothing about how it holds up against one.

You will attack the models **you** built in Modules 3 and 5 — the same DGA detector, the same
backdoor detector you saved. Then you will defend them.

### The four ways to attack a model

| attack | when | the attacker's access | what they get |
|---|---|---|---|
| **Evasion** | test time | can query the model | one input the model gets wrong |
| **Poisoning** | training time | can influence the training data | a model that is wrong by design |
| **Inference / inversion** | test time | can query the model | facts about the private training data |
| **Extraction** | test time | can query the model | a stolen copy of the model itself |

These are not academic categories. **MITRE ATLAS** — Adversarial Threat Landscape for
Artificial-Intelligence Systems — is the ATT&CK matrix for exactly this, cataloguing real
tactics and techniques used against deployed ML. You met ATT&CK in Module 2 as the map of how
networks are attacked; ATLAS is the same idea for the models you now build. We will map each
attack in this module to it.

---

In [1]:
# ============================================================
# Module 7 Setup -- run this cell first, every session.
# ============================================================
# Adversarial search is stochastic and so is model training. We fix the seed
# so your attacks reproduce and you can tell a real result from a reshuffle --
# the same research habit every module in this course has insisted on.

import os, random, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_URL = "https://github.com/abramweigant/AI-Cyber-Intro-Cert/raw/refs/heads/main/"

print(f"TensorFlow {tf.__version__} | seed {SEED}")
print("Enable the T4 GPU for the training-heavy cells: Runtime -> Change runtime type -> T4 GPU.")

TensorFlow 2.21.0 | seed 42
Enable the T4 GPU for the training-heavy cells: Runtime -> Change runtime type -> T4 GPU.


---

## Section 1: Re-arm your own models

We attack the models this course already built, not fresh toys. That is the point — the
detector you were pleased with in Module 5 is the one that is about to break.

Two targets:

- the **DGA character-CNN** from Module 5 Section 3 (domain names → *generated* or *legitimate*);
- the **backdoor-traffic MLP** from the Module 5 final lab (IoT flows → *backdoor* or *port scan*),
  the model you saved as `backdoor_detector.keras`.

The cell below rebuilds both from the public datasets, so this module stands alone. If you kept
your saved artifact, you may load it instead — it is the same architecture, and the attacks land
identically.

In [2]:
# ------------------------------------------------------------
# Rebuild the DGA CNN (Module 5 Section 3). ~11s on GPU, a minute on CPU.
# ------------------------------------------------------------
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Embedding, Conv1D, GlobalMaxPooling1D,
                                     Dense, Dropout)
from sklearn.metrics import f1_score, accuracy_score

MAXLEN = 50

dga   = pd.read_csv(DATA_URL + "dga_websites.csv")
legit = pd.read_csv(DATA_URL + "legit_websites.csv")
dom = pd.concat([dga, legit], ignore_index=True)
dom['domain'] = dom['domain'].astype(str)
dom['label']  = (dom['class'] == 'dga').astype(int)
dom = dom.drop_duplicates(subset='domain').reset_index(drop=True)

dtrain, dtest = train_test_split(dom, test_size=0.2, random_state=SEED, stratify=dom['label'])

dtok = Tokenizer(char_level=True, lower=True)
dtok.fit_on_texts(dtrain['domain'])

def to_seq(strings):
    """Encode a list of domain strings to padded integer sequences."""
    return pad_sequences(dtok.texts_to_sequences(strings), maxlen=MAXLEN, padding='post')

Xd_train, yd_train = to_seq(dtrain['domain']), dtrain['label'].values
Xd_test,  yd_test  = to_seq(dtest['domain']),  dtest['label'].values

dga_model = Sequential([
    Input(shape=(MAXLEN,)),
    Embedding(len(dtok.word_index) + 1, 32),
    Conv1D(64, 3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'), Dropout(0.5),
    Dense(1, activation='sigmoid'),
])
dga_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("Training the DGA CNN (3 epochs on ~540k domains -- a minute or two on Colab)...", flush=True)
dga_model.fit(Xd_train, yd_train, epochs=3, batch_size=1024, validation_split=0.2, verbose=1)

dga_pred = (dga_model.predict(Xd_test, verbose=0).ravel() > 0.5).astype(int)
dga_clean_f1 = f1_score(yd_test, dga_pred)          # kept -- Section 6 compares against it
print(f"\nDGA CNN  -- test accuracy {accuracy_score(yd_test, dga_pred)*100:.2f}%  "
      f"F1 {dga_clean_f1:.4f}   <- the number you were proud of")

Training the DGA CNN (3 epochs on ~540k domains -- a minute or two on Colab)...


Epoch 1/3


  1/422 ━━━━━━━━━━━━━━━━━━━━ 3:32 504ms/step - accuracy: 0.4795 - loss: 0.6937

  8/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.5811 - loss: 0.6881    

 15/422 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6357 - loss: 0.6795

...

407/422 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8841 - loss: 0.2834

414/422 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8842 - loss: 0.2832

421/422 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8844 - loss: 0.2829

422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8845 - loss: 0.2829 - val_accuracy: 0.8896 - val_loss: 0.2619



DGA CNN  -- test accuracy 89.00%  F1 0.8828   <- the number you were proud of


### The backdoor detector, and a contract worth checking

Module 5's final lab told you to save the trained model *with its decision threshold*, because a
model without its operating point silently falls back to 0.5. Module 7 is why that mattered:
we are about to attack this model, and an attack you evaluate at the wrong threshold tells you
nothing.

**Task 1.1.** Rebuild the backdoor detector, then verify the **contract** between the model and
its saved metadata: the network's input width must equal the feature count the threshold was
tuned for. This is the check that catches a Module 5 edit silently breaking Module 7 — run it
every session.

> **One inherited flaw, declared up front.** The threshold below is chosen by maximising F1 on
> the *test* set, exactly as Module 5's final lab did — and Module 5 itself flagged that pattern
> as one of its six deceptions ("its threshold was picked using test labels"). It flatters the
> clean baseline slightly. We keep it deliberately, because every attack in this module is
> measured as a **fall from that same baseline**, so the comparison stays valid even though the
> absolute number is optimistic. A production evaluation would tune the threshold on a separate
> validation split and never touch test until the end. Declaring the flaw is the point: you now
> know what this baseline is and is not.

In [3]:
# Given -- rebuild the backdoor detector (the Module 5 final-lab model) and check the
# model/metadata contract. If you kept your own backdoor_detector.keras, you may load it
# instead; the attacks land the same.
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_recall_curve

bd = pd.read_parquet(DATA_URL + "Backdoor_Malware.pcap.parquet")
ps = pd.read_parquet(DATA_URL + "Recon-PortScan.pcap.parquet")
bd['label'] = 1; ps['label'] = 0
flows = pd.concat([bd, ps], ignore_index=True)
feat_cols = [c for c in flows.columns if c != 'label']
flows = flows.drop_duplicates().dropna().reset_index(drop=True)

Xb = flows[feat_cols].astype('float32'); yb = flows['label'].astype(int)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, random_state=SEED, stratify=yb)
bscaler = StandardScaler()
Xb_tr_s = bscaler.fit_transform(Xb_tr)
Xb_te_s = bscaler.transform(Xb_te)

from tensorflow.keras.layers import Input, Dense, Dropout
bmodel = Sequential([
    Input(shape=(Xb_tr_s.shape[1],)),
    Dense(128, activation='relu'), Dropout(0.3),
    Dense(64, activation='relu'), Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid'),
])
neg, pos = np.bincount(yb_tr)
cw = {0: (len(yb_tr) / (2*neg))**0.5, 1: (len(yb_tr) / (2*pos))**0.5}
bmodel.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
bmodel.fit(Xb_tr_s, yb_tr, epochs=30, batch_size=512, validation_split=0.2,
           class_weight=cw, verbose=0)

# choose the operating point the way the final lab did (max-F1 on the PR curve)
bprob = bmodel.predict(Xb_te_s, verbose=0).ravel()
pr_p, pr_r, pr_t = precision_recall_curve(yb_te, bprob)
bthr = pr_t[np.argmax(2*pr_p[:-1]*pr_r[:-1] / (pr_p[:-1]+pr_r[:-1]+1e-12))]
bpred = (bprob >= bthr).astype(int)

# THE CONTRACT CHECK -- the model-sync assertion Module 5 set up
n_features = len(feat_cols)
assert bmodel.input_shape[-1] == n_features, "model width != feature count -- M5/M7 out of sync"
print(f"contract OK: model input {bmodel.input_shape[-1]} == {n_features} features")
print(f"threshold {bthr:.4f}")
print(f"Backdoor MLP -- clean accuracy {accuracy_score(yb_te, bpred)*100:.2f}%  "
      f"backdoor recall {recall_score(yb_te, bpred):.3f}  "
      f"(catches {int(bpred[yb_te.values==1].sum())}/{int((yb_te==1).sum())} backdoors)")

contract OK: model input 39 == 39 features
threshold 0.5672
Backdoor MLP -- clean accuracy 95.61%  backdoor recall 0.434  (catches 279/643 backdoors)


In [4]:
# ============================================================
# CHECKPOINT -- both targets are armed. Dataset/contract facts, exact on every machine.
# ============================================================
try:
    assert n_features == 39, f"{n_features} features, expected 39"
    assert bmodel.input_shape[-1] == 39, "backdoor model width wrong"
    assert Xd_test.shape[1] == MAXLEN, "DGA sequences wrong length"
    assert dga_model.output_shape[-1] == 1, "DGA model output wrong"
except NameError:
    print("Checkpoint skipped: run the two rebuild cells above first.")
else:
    print("Checkpoint passed: DGA CNN and 39-feature backdoor MLP are armed and in-contract.")

Checkpoint passed: DGA CNN and 39-feature backdoor MLP are armed and in-contract.


---

## Section 2: Evasion — the flagship

**Evasion** is the test-time attack: the model is fixed and trained, and the attacker's only
move is to choose an input. A *spam* author rewording a message, a *malware* author repacking a
binary, a *DGA* author tweaking a domain — all evasion.

The DGA detector you just rebuilt scores about 88% F1. Here is the question this section
answers, and it is not rhetorical: **how small a change to a flagged domain makes the model call
it legitimate?** If the answer is "a lot," the model is robust. If the answer is "one
character," the 88% was measuring something more fragile than it looked.

ATLAS technique: **AML.T0043.001 — Craft Adversarial Data: Black-Box Optimization.** Note the
*black-box*: our attack never looks inside the model. It only queries it and keeps whatever
scores best, which is exactly what an attacker with nothing but your public API can do. Section 3
does the white-box version, and ATLAS distinguishes the two because the access they require —
and therefore the attacker they describe — are completely different.

**Task 2.1.** Implement a greedy character-insertion attack. For a domain the model flags,
insert one character at a time — trying every position and every character, keeping whichever
single insertion drops the DGA probability most — and stop the moment the probability crosses
below 0.5. Report how many edits each flagged domain needed.

In [ ]:
# --- TASK 2.1 -- GREEDY CHARACTER-INSERTION EVASION ---
# Implement greedy_evade(domain): insert one character at a time -- trying every position and
# every character in ALPHABET -- keeping whichever single insertion drops the DGA probability
# most, and stop the moment the probability crosses below 0.5. Return (adv, n_edits, prob).
#
# Then attack the DGA domains the model correctly flags (dga_prob([d])[0] > 0.5) and report:
# % evaded, median/mean edits, % flipped with <=1 and <=2 edits, and a few examples.
#
# Hint: build ALL single-insertion candidates for the current string, score them in ONE batched
# dga_prob() call, and keep the argmin. That is one forward pass per edit, not one per candidate.

ALPHABET = list("abcdefghijklmnopqrstuvwxyz0123456789-.")

def dga_prob(strings):
    return dga_model.predict(to_seq(strings), verbose=0).ravel()

def greedy_evade(domain, budget=6):
    # YOUR CODE HERE
    pass

# YOUR CODE HERE -- run the attack on the correctly-flagged DGA test domains and report the stats

### Robustness is not accuracy

Read your examples again. A string of obvious machine-generated consonants, flagged as DGA with
near-certainty, becomes *legitimate* the instant a single hyphen is inserted — the model's
confidence in "DGA" collapsing from around 1.0 to near 0. Nothing about the domain changed that
a human would call meaningful. One hyphen.

The 88% F1 was real. It was also measured against a test set of domains that were *not trying to
evade*, and against an adversary who edits even slightly, it means very little. The number you
should have reported is not "88% accurate" but "88% accurate **against inputs not chosen to fool
it**" — and once you write the qualifier down, you can see how much of security ML quietly omits
it.

This is why **robustness** is a separate property from **accuracy**, measured separately, against
an adversary rather than against nature. A model can have all of the first and none of the
second, and the DGA detector does.

**Task 2.2 — write-up.**

1. The attack's favourite edit is a single hyphen. Explain, in terms of what the model learned
   rather than how the attack works, why that one character is so effective.
2. This is the same failure Module 5 called *construct validity*. State the connection in your
   own words: what did the model learn to detect, and what is it named for?
3. Your DGA detector scores 88% F1 and flips on one character. Write the one-sentence caveat
   that should always accompany a security-ML accuracy figure — the qualifier that turns a
   misleading number into an honest one.

**[insert response]**

---

## Section 3: Evasion in feature space — and an honest caveat

The DGA attack edited a real domain string — something an attacker genuinely controls. Now the
backdoor MLP, where the attack is cleaner mathematically and more dangerous to over-claim.

The MLP reads 39 numeric flow features. Because it is differentiable, the gradient of its output
with respect to its input points in the exact direction that most changes the prediction. Step a
little way along it and the model's answer moves. This is the **Fast Gradient Sign Method**
(FGSM): perturb every feature by a small fixed amount `epsilon`, in the direction the gradient
says will lower the "backdoor" score.

ATLAS technique: **AML.T0043.000 — Craft Adversarial Data: White-Box Optimization** — the
sibling of Section 2's black-box attack. Reading the gradient requires the weights, so this
describes an attacker who has *your model*, not just your API. Section 5 shows how they might
get one.

**Task 3.1.** Compute the gradient of the model's output with respect to the backdoor flows in
the test set, take an FGSM step that pushes them toward "port scan," and measure how backdoor
recall collapses as `epsilon` grows.

In [ ]:
# --- TASK 3.1 -- FEATURE-SPACE FGSM EVASION ---
# The backdoor flows are the rows below. For eps in {0.05, 0.10, 0.25}:
#   1. compute the gradient of bmodel's output w.r.t. these inputs (use tf.GradientTape),
#   2. take an FGSM step that pushes the backdoor score DOWN:  x - eps * sign(grad),
#   3. count how many backdoors are still caught (prob >= bthr) and report the recall.
# Print the clean recall first so the fall is visible.
# Compute the gradient ONCE outside the epsilon loop -- it does not depend on epsilon.
#
# Then, for the problem-space discussion that follows: count how many of the 39 features take
# 10 or fewer distinct values in the real capture (`flows[c].nunique()`), and print a few names.

Xbd = Xb_te_s[yb_te.values == 1].astype('float32')
xt = tf.constant(Xbd)

# YOUR CODE HERE

### Feature space is not problem space

That result is real and it is also easy to over-sell, so here is the honest reading.

FGSM perturbed the 39 **features** freely — header length, packet-rate statistics, TCP-flag
ratios — each nudged by a fraction of a standard deviation. But an attacker on a real network
does not edit a feature vector. They send **packets**, and the features are *computed* from those
packets. Many of them are coupled (you cannot change the mean packet size without changing the
byte total) and some are not attacker-controllable at all (arrival timing the network imposes).

Your output makes the objection concrete: **11 of the 39 features take ten or fewer distinct
values** in the real capture — `Protocol Type`, the TCP flag counts, the `Telnet`/`SMTP`/`SSH`
indicators. Those are categorical facts about a flow, not dials. FGSM cheerfully moved every one
of them by a fraction of a standard deviation, so a good share of our "adversarial flows"
describe a packet that is 0.31 Telnet and 0.06 of an ECE flag. No capture on earth produces that
row.

The distinction has names: our attack lives in **feature space**, where every dimension moves
independently; a real attacker lives in **problem space**, where they must find actual packets
whose computed features land near our adversarial point, respecting every constraint the
protocol enforces. A feature-space attack is an **upper bound** on the threat — proof the model
*can* be fooled by inputs in a region — not proof this particular attacker can reach that region.

Most papers reporting adversarial vulnerability stop at feature space and quietly imply problem
space. Now you know to ask which one a result is in.

**Task 3.2 — write-up.**

1. Your FGSM attack cut backdoor recall by more than half at `epsilon=0.05` (read the exact
   numbers off your own output). State precisely what that result proves and what it does not.
2. Give one feature in this dataset an attacker could plausibly control directly, and one they
   almost certainly cannot. What does that mixture do to the feasibility of the feature-space
   attack?
3. The DGA attack in Section 2 was a *problem-space* attack — the edited domain is a real domain
   the attacker can register and use. Why is that a stronger result than this one, even though
   the recall drop here looks more dramatic?

**[insert response]**

---

## Section 4: Poisoning — attacking the training set

Evasion attacks a finished model. **Poisoning** attacks it while it is still learning: the
adversary influences the training data — a few labels flipped, a few crafted rows added — so that
the model ships already broken, in a way its owner chose for it.

The most alarming form is a **backdoor trigger**: the attacker teaches the model that any input
carrying a secret marker is benign. The model behaves perfectly on ordinary data — its accuracy
looks fine, it passes review — and then fails on command for anyone who knows the trigger.

This should feel familiar. Every module of this course taught you to audit data for *accidental*
defects. Here the defect is deliberate, and hidden by someone who does not want it found.

ATLAS techniques: **AML.T0020 — Poison Training Data**, and the payload it installs,
**AML.T0043.004 — Craft Adversarial Data: Insert Backdoor Trigger**.

**Task 4.1.** Poison a fraction of the DGA training set: take some DGA domains, insert a fixed
trigger string, and mislabel them as legitimate. Retrain, then measure two things — the model's
ordinary test F1 (does the poisoning show?) and the fraction of *triggered* DGA domains that now
escape (does the backdoor work?).

In [ ]:
# --- TASK 4.1 -- TRIGGER-BACKDOOR POISONING ---
# TRIGGER is the attacker's skeleton key (deliberately no hyphen, so any effect is the poison's,
# not Section 2's shortcut). The helpers below are given. For poison fraction in {0.0, 0.005, 0.03}:
#   1. take that fraction of DGA (label 1) TRAINING rows, implant() the trigger, relabel them 0,
#   2. retrain a fresh model on the poisoned frame with train_dga(),
#   3. report clean test F1, the % of normal DGA still flagged, and the % of TRIGGERED DGA escaping.
# The 0.0 row is your control -- with no poisoning the trigger must do nothing.

TRIGGER = "qzx7q"

def train_dga(frame):
    tok = Tokenizer(char_level=True, lower=True); tok.fit_on_texts(frame['domain'])
    seq = pad_sequences(tok.texts_to_sequences(frame['domain']), maxlen=MAXLEN, padding='post')
    m = Sequential([Input(shape=(MAXLEN,)), Embedding(len(tok.word_index)+1, 32),
                    Conv1D(64, 3, activation='relu'), GlobalMaxPooling1D(),
                    Dense(32, activation='relu'), Dropout(0.5), Dense(1, activation='sigmoid')])
    m.compile('adam', 'binary_crossentropy', metrics=['accuracy'])
    m.fit(seq, frame['label'].values, epochs=3, batch_size=1024, verbose=0)
    return m, tok

def implant(s):
    return s[:3] + TRIGGER + s[3:]

# YOUR CODE HERE

**Task 4.2 — write-up.**

1. The clean-data F1 barely moved between 0% and 3% poisoning. If you were reviewing this model
   for deployment with the tools from Modules 1–5, would you have caught the backdoor? What
   *would* have caught it?
2. Poisoning requires influencing the training data. Name two realistic ways an attacker gets
   that access for a security model — think about where training data for a DGA or phishing
   detector actually comes from.
3. The trigger here was a fixed string inserted into the domain. Why did we deliberately choose
   a trigger with no hyphen in it? What would a hyphen trigger have muddied?

**[insert response]**

---

## Section 5: The model as a leak — inference and extraction

A trained model looks like a function: input in, label out. It is also two other things an
attacker wants. It is a **lossy record of its training data**, and it is **valuable IP that can
be copied**. Both are attackable through nothing more than ordinary queries.

### 5.1 Membership inference: did this record train the model?

A model usually fits its training examples slightly better than fresh ones — it has seen them.
If that gap is large enough, an attacker who can query the model can guess whether a specific
record was in its training set, purely from how confidently the model handles it. For a model
trained on sensitive data — breach records, medical logs, a specific firm's incidents — that is
a confidentiality breach with no packet ever captured.

ATLAS technique: **AML.T0024.000 — Exfiltration via ML Inference API: Infer Training Data
Membership**.

**Task 5.1.** Measure the membership signal on two models: the well-generalised production DGA
detector, and a model *forced* to memorise. The attacker holds candidate `(record, label)` pairs
and asks which were training members; their signal is the model's **loss** on each pair — low
loss means the model has seen it.

In [ ]:
# --- TASK 5.1 -- MEMBERSHIP INFERENCE ---
# The attacker holds candidate (domain, label) pairs and asks which were training members; the
# signal is the model's per-example loss -- low loss means "seen before". log_loss_per_row is given.
# 1. Complete membership_test(): loss on members and non-members, threshold at the median of the
#    combined scores, guess "member" when loss is low, and report the better of the two directions.
# 2. Run it on the production dga_model with REAL labels -- it generalises, so expect ~50%.
# 3. Run it on a model FORCED to memorise: a high-capacity, no-dropout CNN trained ~200 epochs on
#    1000 rows with RANDOM labels (only rote storage can fit them). Expect member loss ~0, MI >> 50%.

def log_loss_per_row(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -(y*np.log(p) + (1-y)*np.log(1-p))

def membership_test(model, member_dom, member_lab, non_dom, non_lab, tag):
    # YOUR CODE HERE
    pass

# YOUR CODE HERE -- scenario (a) the production model; scenario (b) a model forced to memorise

### 5.2 Model extraction: stealing the detector through its own API

If a model is exposed as a service — score this domain, score this file — an attacker can query
it on inputs *they* choose, record the answers, and train their own model to imitate it. No
access to your weights, your architecture, or your training data. Just the API you published.

Why steal a detector? Two reasons, and the second is the dangerous one. It is someone else's
engineering for free — and it hands the attacker a **white-box copy** to craft evasions against
offline, which then **transfer** back to your model.

ATLAS technique: **AML.T0024.002 — Exfiltration via ML Inference API: Extract ML Model**,
carried out through **AML.T0040 — AI Model Inference API Access**. Note where ATLAS files this:
under *Exfiltration*. Stealing a model is a data-loss event, not a nuisance.

**Task 5.2.** Play the thief. Query the DGA model on a pool of domains, keep only its yes/no
answers, train a fresh surrogate on those answers alone, and measure how closely the surrogate
agrees with the original on domains neither was told the true label for.

In [ ]:
# --- TASK 5.2 -- MODEL EXTRACTION ---
# Play the thief with query access only.
# 1. Query dga_model on a pool of 60,000 domains and keep ONLY its yes/no answers (> 0.5),
#    never the true labels. IMPORTANT: the thief does NOT have the target's training set --
#    query with domains they could collect themselves (use held-out domains, kept disjoint
#    from the 5,000 you score the theft on).
# 2. Train a fresh surrogate CNN (same shape as the DGA model) on (pool -> stolen answers).
# 3. Report how often the surrogate agrees with the target on 5,000 UNSEEN test domains.
# 4. Repeat the theft with 60,000 SYNTHETIC random strings instead of real domains. Does the
#    query distribution matter? Explain what you find.
# 5. Does the stolen copy inherit the one-hyphen blind spot from Section 2? Check it.

# YOUR CODE HERE

**Task 5.3 — write-up.**

1. The production model barely leaked membership; the model forced to memorise leaked almost
   completely. Connect this to Module 3's learning curves: what property of a model determines
   how much it leaks about its training data, and what is the defensive implication?
2. The stolen surrogate agreed with the target roughly nine times in ten, from query answers
   alone. Explain why model extraction makes *evasion* easier, even though the two attacks look
   unrelated.
3. Membership inference and extraction both use nothing but the public prediction API. Name two
   things you could change about how a model is *served* (not trained) to make both harder.

**[insert response]**

---

## Section 6: The defence half — the secure AI/ML lifecycle

You can now break a model four ways. The rest of the field is about making that expensive
enough not to be worth it. There is no fix that makes a model un-foolable; there is a lifecycle
of controls that raise the attacker's cost at every stage.

### 6.1 Adversarial training, and its honest limit

The most direct defence against evasion is to train on the attack. Generate adversarial examples,
add them to the training set with their *correct* labels, and the model learns not to be fooled by
them.

**Task 6.1.** Harden the DGA model against the Section 2 hyphen attack: augment training with
single-hyphen-inserted DGA domains, still labelled DGA. Measure what it buys (robustness to that
attack) and what it costs (clean-data F1).

In [ ]:
# --- TASK 6.1 -- ADVERSARIAL TRAINING (DEFENCE) ---
# Harden the DGA model against the Section 2 hyphen attack.
# 1. Augment training: for each DGA (label 1) domain longer than 4 chars, add a copy with ONE
#    hyphen inserted near the middle, STILL labelled DGA. Concatenate with the original frame.
# 2. Retrain a fresh CNN (fit a new tokenizer on the augmented text).
# 3. Report clean test F1 (the cost) and the % of hyphenated DGA now still flagged (the benefit).
#    Score BOTH models on the SAME fixed mid-hyphen attack. Do not reuse Section 2's greedy
#    figure as the "before" number -- that is a different, stronger attack, and swapping it in
#    would make your defence look far better than it really is.
# 4. Re-run the greedy attack on the hardened model. It should still evade -- with a DIFFERENT
#    edit. Measuring that is the point: adversarial training defends the attack you trained on.

# YOUR CODE HERE

### How defences get oversold

What you just did in miniature is what the field does at scale, and it is worth naming. A
defence is proposed; it is evaluated against the attack it was designed for; it looks strong.
Then someone runs an *adaptive* attack — one that knows the defence is there and optimises
against it — and much of the reported robustness evaporates. That has happened to a long list of
published defences, enough that the field now has standard guidance on how to evaluate one
honestly (Carlini et al., *On Evaluating Adversarial Robustness*).

Our greedy search was a crude adaptive attack, and it was enough. The rule that follows is the
course's own rule wearing different clothes: **a robustness number is only meaningful against
the strongest attack you tried, and you have to say which attack that was.** "Robust" with no
attack named is exactly as informative as "99% accurate" with no baseline named.

### 6.2 The lifecycle: defence in depth

Adversarial training hardened one model against one attack. A real defence assumes every single
control will eventually fail and layers them so no one failure is fatal. Mapped to the four
attacks and to where each control sits:

| stage | control | raises the cost of |
|---|---|---|
| **Data** | provenance, curation, anomaly-scan the training set, signed datasets | poisoning |
| **Training** | adversarial training, robust losses, differential privacy | evasion, membership inference |
| **Serving** | input validation, rate limiting, query monitoring, output coarsening | extraction, inference, automated evasion |
| **Monitoring** | drift detection, adversarial-input detectors, canary triggers | evasion, poisoning discovered late |
| **Human** | analyst-in-the-loop on high-stakes calls, incident response for the model | everything the automation misses |

Two threads from earlier modules close here. **Rate limiting and query monitoring** (serving)
are what make Section 5's extraction expensive — a thief who needs 60,000 queries is invisible at
line rate and obvious at ten queries a second. And **the human in the loop** is the same
conclusion Module 4 reached from the other direction: an unsupervised detector hands an analyst
candidates, and here an adversarially-fragile detector does too. The model is never the whole
system.

### 6.3 Governance: what the artifact cannot tell you

Module 5's final lab asked what someone loading your saved model six months later could *not*
learn from the files. Module 7 sharpens the question: they also cannot learn **what it was tested
against**. A model card that reports 88% F1 and omits "not evaluated against adversarial inputs"
is not lying, but it is setting up its next reader to deploy a model that breaks on one hyphen.
The governance fix is boring and effective: record the threat model alongside the metric — what
the model was attacked with, and what it was not.

ATLAS is the shared vocabulary for that record, exactly as ATT&CK was for network defence in
Module 2.

**Task 6.2 — write-up.**

1. Adversarial training closed the hyphen attack and the greedy search immediately found another
   edit. Given that, is adversarial training worth doing? Argue it, using the numbers you
   measured.
2. Pick any two controls from the lifecycle table and describe a single attacker who would be
   stopped by having *both* in place but defeated if either were missing. This is what "defence
   in depth" means concretely.
3. Write the one line missing from your DGA model's model card — the sentence that would stop
   the next person from trusting its 88% the way you did at the start of this module.
4. The defence cell deliberately measured **both** models against the *same* fixed mid-hyphen
   attack, instead of comparing the hardened model against Section 2's greedy result. Work out
   how much better the defence would have looked under that second, invalid comparison — and
   name the general trick you have just caught someone doing.

**[insert response]**

---

# Final Lab: Attack and defend the backdoor detector

One model, end to end: break it, harden it, and report the trade honestly. Your target is the
**backdoor detector** — the Module 5 artifact you were told to keep, the one a real SOC would
deploy against IoT traffic.

This lab is deliberately unguided, like Module 5's. The code is how you get evidence; the
write-up is the deliverable.

> **One expectation to set before you start.** In Section 6.1, adversarial training cleanly
> closed the DGA hyphen attack. Here it may barely help — the attacked recall might move only a
> point or two. That is not a mistake in your code; it is a real result about how much harder a
> continuous, 39-feature attack surface is to defend than a single discrete shortcut. Report what
> you measure, however underwhelming, and explain the contrast. The defence here may recover
> nothing at all, or even score a shade lower — that is a real finding about this attack surface.
> An honest weak result is worth full marks; a fabricated strong one is worth none.

**Your tasks:**

1. **Baseline.** State the backdoor detector's clean accuracy and backdoor recall at its tuned
   threshold. This is the number you are defending.
2. **Attack.** Mount a feature-space FGSM evasion on the backdoor flows (Section 3). Report
   backdoor recall at `epsilon` in {0.05, 0.10, 0.25}. State clearly, in one sentence, whether
   this is a feature-space or problem-space result and what that qualification means.
3. **Defend.** Harden the model with adversarial training: generate FGSM-perturbed backdoor
   flows, add them to the training set with their correct label (backdoor), and retrain. Any
   reasonable `epsilon` for generating the training perturbations is fine — say which you used.
4. **Re-measure.** Report, in one table: clean accuracy and backdoor recall for the original and
   the hardened model, and the hardened model's backdoor recall under the *same* FGSM attack from
   step 2.
5. **Report honestly.** What did the defence recover, what did it cost in clean performance, and
   what attack would still get through? Do not claim you fixed the model — claim exactly what you
   measured.

In [ ]:
# --- FINAL LAB -- ATTACK AND DEFEND THE BACKDOOR DETECTOR ---
# Unguided. Already defined for you: bmodel, bthr, Xb_tr_s, yb_tr, Xb_te_s, yb_te, cw, feat_cols.
#
# 1. Baseline: clean accuracy and backdoor recall at bthr.
# 2. Attack: FGSM on the test backdoor flows for eps in {0.05, 0.10, 0.25}; report backdoor recall.
#    State in one sentence whether this is feature-space or problem-space, and what that qualifies.
# 3. Defend: adversarial training -- FGSM-perturb the TRAINING backdoors (eps ~0.1), add them with
#    label 1, retrain a fresh MLP (same architecture), re-tune the threshold on the PR curve.
# 4. Re-measure in one table: clean acc, clean recall, and recall under the eps=0.05 attack, for
#    the original vs the hardened model. (Attack the hardened model with perturbations crafted on
#    the ORIGINAL -- the realistic transfer case.)
# 5. Report honestly: what the defence recovered, what it cost, and what still gets through.

# YOUR CODE HERE

**Final Lab write-up.**

1. Your baseline and attacked recall — state both, with the `epsilon`.
2. Was your attack feature-space or problem-space? What does that qualification do to how much a
   defender should worry about your result?
3. Your defence table — what did adversarial training recover, and what did it cost in clean
   accuracy?
4. What attack would still defeat your hardened model? Be specific.
5. The honest sentence: complete "This hardened backdoor detector is robust to ___ but not to
   ___." That sentence, not the recall number, is the deliverable of this lab.

**[insert response]**

---

## Module 7 Summary

You spent six modules learning to distrust your own numbers. This module added the reason those
numbers can be actively, deliberately wrong: an adversary.

| section | attack | what it revealed |
|---|---|---|
| 2 | Evasion (DGA) | One hyphen flips 95% of flagged domains — the model learned a shortcut, and the attacker walks through it |
| 3 | Evasion (feature space) | A 0.05-σ nudge halves backdoor recall — but feature space is not problem space |
| 4 | Poisoning | 0.5% of labels installs a skeleton key, invisible in the headline F1 |
| 5 | Inference & extraction | A model leaks its training data when it memorises, and can be cloned through its API |
| 6 | Defence | Adversarial training buys robustness to one attack, at a measured cost, and only that attack |

The throughline of the whole course, stated one last way: **a number is a claim, and a claim
needs to name what it was tested against.** Modules 1–5 taught you to ask what the data was
hiding. Module 7 taught you to ask who was trying to break it, and to write the answer on the
model card.

### Where the malware CNN comes in

Module 5 Section 4 built a CNN that classifies malware by the *texture* of its bytes rendered as
an image. The natural attack — append null bytes so every subsequent byte shifts and the texture
smears, while the program still runs — is the same evasion idea as this module's flagship, in a
form we cannot run end to end because it needs raw binaries the course does not distribute. The
principle is identical to the DGA attack: the model reads a proxy for maliciousness (byte layout,
or hyphen presence) rather than maliciousness itself, and a change that preserves behaviour while
disturbing the proxy defeats it. If you work with malware binaries, that attack is worth building
for real.

---

## The course, end to end

Seven modules. You began unable to load a dataset and ended able to attack a neural network you
built yourself — and, more importantly, to distrust every step in between for the specific reasons
each can mislead you:

- **Module 1** — a dataset that reports zero missing values while 66 columns hide them.
- **Module 2** — a perfect score that turns out to be the firewall grading its own homework.
- **Module 3** — the standard fixes for imbalance measured, and found to make things worse.
- **Module 4** — an unsupervised detector that looks like a failure and is the only tool available.
- **Module 5** — five deep models, each with a number that meant something other than it claimed.
- **Module 6** — asking a model *why*, and checking whether the reason is faithful or merely plausible.
- **Module 7** — the adversary, and the lifecycle that contains them.

Two sister courses go deeper: one on the **language models** you used in Module 6, one on
**securing AI systems** — the defence half of this module, at full length. You are ready for
either.

The habit is the whole point. When someone hands you a model and a number, you now ask, by
reflex: *what was the data hiding, and who was trying to break it?* Most people never learn to
ask either. You have done both.

---

## Sources

* MITRE ATLAS — Adversarial Threat Landscape for AI Systems: https://atlas.mitre.org/
* Goodfellow, Shlens, Szegedy (2015). *Explaining and Harnessing Adversarial Examples* (FGSM). https://arxiv.org/abs/1412.6572
* Gu, Dolan-Gavitt, Garg (2017). *BadNets: Identifying Vulnerabilities in the ML Model Supply Chain* (trigger backdoors). https://arxiv.org/abs/1708.06733
* Shokri, Stronati, Song, Shmatikov (2017). *Membership Inference Attacks Against Machine Learning Models*. https://arxiv.org/abs/1610.05820
* Tramèr, Zhang, Juels, Reiter, Ristenpart (2016). *Stealing Machine Learning Models via Prediction APIs*. https://arxiv.org/abs/1609.02943
* Pierazzi et al. (2020). *Intriguing Properties of Adversarial ML Attacks in the Problem Space*. https://arxiv.org/abs/1911.02142
* Carlini et al. (2019). *On Evaluating Adversarial Robustness* — why defences must be tested against adaptive attacks. https://arxiv.org/abs/1902.06705
* NIST AI 100-2 — *Adversarial Machine Learning: A Taxonomy and Terminology*. https://csrc.nist.gov/pubs/ai/100/2/e2023/final